# Association Rule Mining Assignment

This notebook contains the full workflow for the assignment:
1. load and inspect the data,
2. preprocess grocery orders into transactions,
3. apply Apriori to discover frequent itemsets and association rules,
4. analyze the effect of support and confidence thresholds,
5. explore a short time-based extension.

## 1. Imports and file paths

In [1]:
from pathlib import Path
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Resolve project paths.
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data"

orders_path = DATA_DIR / "orders.csv"
order_products_path = DATA_DIR / "order_products.csv"
products_path = DATA_DIR / "products.csv"
aisles_path = DATA_DIR / "aisles.csv"
departments_path = DATA_DIR / "departments.csv"

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)

Base directory: c:\Users\Admin\Storage\Akademik\Dersler\Erasmus Classes\Data Mining\Assignment1
Data directory: c:\Users\Admin\Storage\Akademik\Dersler\Erasmus Classes\Data Mining\Assignment1\data


## 2. Load the data

In [2]:
# Load the smaller tables fully.
orders = pd.read_csv(orders_path)
products = pd.read_csv(products_path)
aisles = pd.read_csv(aisles_path)
departments = pd.read_csv(departments_path)

print("Orders:", orders.shape)
print("Products:", products.shape)
print("Aisles:", aisles.shape)
print("Departments:", departments.shape)

Orders: (3421083, 6)
Products: (49688, 4)
Aisles: (134, 2)
Departments: (21, 2)


In [3]:
# Preview the large order-products file before using it in the full pipeline.
order_products_preview = pd.read_csv(order_products_path, nrows=10)
order_products_preview

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1
5,1,13176,6,0
6,1,47209,7,0
7,1,22035,8,1
8,2,33120,1,1
9,2,28985,2,1


## 3. Sample orders and build transactions

In [4]:
# Sample a manageable subset of orders for association rule mining.
sample_size = 50000
sampled_orders = orders.sample(n=sample_size, random_state=42)
sampled_order_ids = set(sampled_orders["order_id"])

print("Sampled orders:", sampled_orders.shape)

Sampled orders: (50000, 6)


In [5]:
# Keep only products that belong to the sampled orders.
order_products = pd.read_csv(order_products_path)
order_products_subset = order_products[
    order_products["order_id"].isin(sampled_order_ids)
]

print("Filtered order_products:", order_products_subset.shape)

Filtered order_products: (495774, 4)


In [6]:
# Add product metadata.
order_products_subset = order_products_subset.merge(
    products,
    on="product_id"
)

# Add aisle names to make items easier to interpret.
order_products_subset = order_products_subset.merge(
    aisles,
    on="aisle_id"
)

order_products_subset.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle
0,47,16797,1,1,Strawberries,24,4,fresh fruits
1,47,39275,2,1,Organic Blueberries,123,4,packaged vegetables fruits
2,47,43352,3,1,Raspberries,32,4,packaged produce
3,47,46041,4,0,Beef Franks,106,12,hot dogs bacon sausage
4,47,29223,5,0,Golden Pineapple,24,4,fresh fruits


In [7]:
# Build transactions at the order level and remove duplicate aisles
# within the same transaction.
transactions = order_products_subset.groupby("order_id")["aisle"].apply(
    lambda x: sorted(set(x))
)

print("Total transactions:", len(transactions))
transactions.iloc[:5]

Total transactions: 48938


order_id
47     [fresh fruits, hot dogs bacon sausage, package...
322    [fresh herbs, fresh vegetables, nuts seeds dri...
360    [baking ingredients, condiments, fresh dips ta...
393    [breakfast bars pastries, candy chocolate, fre...
441    [asian foods, body lotions soap, cleaning prod...
Name: aisle, dtype: object

## 4. Encode transactions

In [8]:
# Convert the transaction lists into a one-hot encoded matrix.
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_transactions = pd.DataFrame(te_array, columns=te.columns_)

df_transactions.head()

,air fresheners candles,asian foods,baby accessories,baby bath body care,baby food formula,bakery desserts,baking ingredients,baking supplies decor,beauty,beers coolers,...,spreads,tea,tofu meat alternatives,tortillas flat bread,trail mix snack mix,trash bags liners,vitamins supplements,water seltzer sparkling water,white wines,yogurt
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,False,False,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 5. Frequent itemsets and initial rules

In [9]:
# Discover frequent itemsets with a first support threshold.
frequent_itemsets = apriori(
    df_transactions,
    min_support=0.01,
    use_colnames=True
)

frequent_itemsets.sort_values("support", ascending=False).head(10)

,support,itemsets
32,0.556030,frozenset({fresh fruits})
35,0.444951,frozenset({fresh vegetables})
68,0.367935,frozenset({packaged vegetables fruits})
544,0.317463,"frozenset({fresh fruits, fresh vegetables})"
574,0.270015,"frozenset({packaged vegetables fruits, fresh f..."
94,0.261842,frozenset({yogurt})
58,0.241653,frozenset({milk})
642,0.234746,"frozenset({packaged vegetables fruits, fresh v..."
65,0.231252,frozenset({packaged cheese})
93,0.192264,frozenset({water seltzer sparkling water})


In [10]:
# Generate association rules using confidence as the main metric.
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.3
)

rules.sort_values("confidence", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
3074,"frozenset({fresh herbs, fresh fruits, canned j...",frozenset({fresh vegetables}),0.014447,0.444951,0.013629,0.943423,2.120286,1.0,0.007201,9.810495,0.536111,0.030575,0.898068,0.487027
3122,"frozenset({fresh herbs, packaged vegetables fr...",frozenset({fresh vegetables}),0.011402,0.444951,0.010687,0.937276,2.106471,1.0,0.005614,8.849071,0.531331,0.023980,0.886994,0.480647
6111,"frozenset({yogurt, packaged vegetables fruits,...",frozenset({fresh fruits}),0.016817,0.556030,0.015673,0.931956,1.676090,1.0,0.006322,6.524772,0.410273,0.028129,0.846738,0.480072
5875,"frozenset({yogurt, packaged vegetables fruits,...",frozenset({fresh fruits}),0.016143,0.556030,0.014999,0.929114,1.670978,1.0,0.006023,6.263147,0.408137,0.026919,0.840336,0.478044
4099,"frozenset({fresh herbs, fresh fruits, soup bro...",frozenset({fresh vegetables}),0.012546,0.444951,0.011647,0.928339,2.086385,1.0,0.006065,7.745460,0.527318,0.026124,0.870892,0.477258
6094,"frozenset({yogurt, packaged cheese, fresh dips...",frozenset({fresh fruits}),0.011545,0.556030,0.010687,0.925664,1.664773,1.0,0.004268,5.972453,0.403981,0.019191,0.832565,0.472442
6408,"frozenset({yogurt, packaged vegetables fruits,...",frozenset({fresh fruits}),0.012056,0.556030,0.011157,0.925424,1.664341,1.0,0.004453,5.953233,0.404033,0.020033,0.832024,0.472745
6130,"frozenset({fresh herbs, packaged vegetables fr...",frozenset({fresh vegetables}),0.016511,0.444951,0.015244,0.923267,2.074988,1.0,0.007897,7.233545,0.526767,0.034162,0.861755,0.478763
6777,"frozenset({packaged vegetables fruits, yogurt,...",frozenset({fresh fruits}),0.017553,0.556030,0.016204,0.923166,1.660282,1.0,0.006444,5.778336,0.404798,0.029072,0.826940,0.476155
5100,"frozenset({fresh herbs, packaged vegetables fr...",frozenset({fresh vegetables}),0.011811,0.444951,0.010891,0.922145,2.072466,1.0,0.005636,7.129299,0.523668,0.024427,0.859734,0.473311


## 6. Threshold analysis

In [ ]:
# Compare how the number of itemsets and rules changes
# with different minimum support values.
support_results = []

for support in [0.02, 0.01, 0.005]:
    frequent_itemsets_tmp = apriori(
        df_transactions,
        min_support=support,
        use_colnames=True,
        max_len=2  # I restricted length here to keep this experiment fast. Otherwise it takes so long to compile because 0.005 is already a low min support value.
    )

    rules_tmp = association_rules(
        frequent_itemsets_tmp,
        metric="confidence",
        min_threshold=0.3
    )

    support_results.append({
        "min_support": support,
        "num_frequent_itemsets": len(frequent_itemsets_tmp),
        "num_rules": len(rules_tmp)
    })

support_results

[{'min_support': 0.02, 'num_frequent_itemsets': 420, 'num_rules': 266},
 {'min_support': 0.01, 'num_frequent_itemsets': 930, 'num_rules': 378},
 {'min_support': 0.005, 'num_frequent_itemsets': 1747, 'num_rules': 459}]

In [12]:
# Compare how the number of rules changes with different
# minimum confidence values.
confidence_results = []

frequent_itemsets_base = apriori(
    df_transactions,
    min_support=0.01,
    use_colnames=True
)

for confidence in [0.2, 0.3, 0.5, 0.7]:
    rules_tmp = association_rules(
        frequent_itemsets_base,
        metric="confidence",
        min_threshold=confidence
    )

    confidence_results.append({
        "min_confidence": confidence,
        "num_rules": len(rules_tmp)
    })

confidence_results

[{'min_confidence': 0.2, 'num_rules': 9753},
 {'min_confidence': 0.3, 'num_rules': 6796},
 {'min_confidence': 0.5, 'num_rules': 3647},
 {'min_confidence': 0.7, 'num_rules': 1598}]

## 7. Final product-category rules

In [13]:
# Create the final rule set used for interpretation in the report.
frequent_itemsets_final = apriori(
    df_transactions,
    min_support=0.01,
    use_colnames=True,
)

rules_final = association_rules(
    frequent_itemsets_final,
    metric="confidence",
    min_threshold=0.3
)

# Keep only one-item antecedents and one-item consequents
# to make the rules easier to interpret.
rules_single = rules_final[
    (rules_final["antecedents"].apply(len) == 1) &
    (rules_final["consequents"].apply(len) == 1)
].copy()

# Keep rules with acceptable support, confidence, and lift.
rules_single_filtered = rules_single[
    (rules_single["support"] >= 0.01) &
    (rules_single["confidence"] >= 0.3) &
    (rules_single["lift"] > 1.0)
].copy()

rules_single_filtered.sort_values(["lift", "confidence"], ascending=[False, False]).head(15)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
123,frozenset({pasta sauce}),frozenset({dry pasta}),0.062303,0.070518,0.019555,0.313873,4.450981,1.0,0.015162,1.354680,0.826846,0.172650,0.261818,0.295592
81,frozenset({preserved dips spreads}),frozenset({chips pretzels}),0.031877,0.170420,0.014488,0.454487,2.666870,1.0,0.009055,1.520735,0.645609,0.077141,0.342423,0.269750
71,frozenset({fresh dips tapenades}),frozenset({chips pretzels}),0.099718,0.170420,0.036066,0.361680,2.122292,1.0,0.019072,1.299631,0.587384,0.154081,0.230551,0.286656
285,frozenset({granola}),frozenset({yogurt}),0.028465,0.261842,0.015652,0.549892,2.100096,1.0,0.008199,1.639960,0.539179,0.056990,0.390229,0.304835
69,frozenset({cookies cakes}),frozenset({chips pretzels}),0.058114,0.170420,0.020495,0.352672,2.069434,1.0,0.010591,1.281546,0.548661,0.098517,0.219693,0.236468
80,frozenset({popcorn jerky}),frozenset({chips pretzels}),0.045486,0.170420,0.015918,0.349955,2.053489,1.0,0.008166,1.276189,0.537472,0.079595,0.216417,0.221680
18,frozenset({lunch meat}),frozenset({bread}),0.106277,0.162777,0.035167,0.330898,2.032825,1.0,0.017867,1.251263,0.568491,0.150358,0.200807,0.273471
372,frozenset({tofu meat alternatives}),frozenset({soy lactosefree}),0.032572,0.170542,0.010994,0.337516,1.979073,1.0,0.005439,1.252041,0.511369,0.057222,0.201304,0.200989
330,frozenset({pasta sauce}),frozenset({packaged cheese}),0.062303,0.231252,0.028505,0.457527,1.978480,1.0,0.014098,1.417118,0.527422,0.107548,0.294342,0.290396
76,frozenset({fruit vegetable snacks}),frozenset({chips pretzels}),0.046365,0.170420,0.015489,0.334068,1.960265,1.0,0.007588,1.245743,0.513682,0.076947,0.197266,0.212478


In [14]:
# Alternative views of the same filtered rules.
rules_single_filtered.sort_values("confidence", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
200,frozenset({fresh herbs}),frozenset({fresh vegetables}),0.092648,0.444951,0.077813,0.839876,1.887572,1.0,0.036589,3.466382,0.518232,0.169237,0.711515,0.507378
52,frozenset({canned jarred vegetables}),frozenset({fresh vegetables}),0.075381,0.444951,0.057910,0.768230,1.726550,1.0,0.024369,2.394827,0.455118,0.125232,0.582433,0.449190
143,frozenset({fresh herbs}),frozenset({fresh fruits}),0.092648,0.556030,0.070150,0.757168,1.361739,1.0,0.018635,1.828301,0.292770,0.121256,0.453044,0.441665
240,frozenset({poultry counter}),frozenset({fresh vegetables}),0.037864,0.444951,0.028669,0.757151,1.701650,1.0,0.011821,2.285569,0.428562,0.063127,0.562472,0.410791
224,frozenset({meat counter}),frozenset({fresh vegetables}),0.020434,0.444951,0.015469,0.757000,1.701312,1.0,0.006376,2.284153,0.420817,0.034381,0.562201,0.395882
5,frozenset({baby food formula}),frozenset({fresh fruits}),0.045690,0.556030,0.033594,0.735242,1.322305,1.0,0.008188,1.676886,0.255415,0.059130,0.403657,0.397829
175,frozenset({packaged vegetables fruits}),frozenset({fresh fruits}),0.367935,0.556030,0.270015,0.733866,1.319832,1.0,0.065432,1.668223,0.383391,0.412899,0.400560,0.609739
212,frozenset({grains rice dried goods}),frozenset({fresh vegetables}),0.041522,0.444951,0.030385,0.731791,1.644657,1.0,0.011910,2.069468,0.408951,0.066622,0.516784,0.400040
252,frozenset({tofu meat alternatives}),frozenset({fresh vegetables}),0.032572,0.444951,0.023806,0.730866,1.642577,1.0,0.009313,2.062351,0.404372,0.052468,0.515116,0.392184
151,frozenset({frozen produce}),frozenset({fresh fruits}),0.124361,0.556030,0.090809,0.730200,1.313239,1.0,0.021660,1.645555,0.272400,0.154022,0.392302,0.446758


In [15]:
rules_single_filtered.sort_values("support", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
144,frozenset({fresh fruits}),frozenset({fresh vegetables}),0.556030,0.444951,0.317463,0.570946,1.283166,1.0,0.070057,1.293657,0.497055,0.464454,0.226998,0.642212
145,frozenset({fresh vegetables}),frozenset({fresh fruits}),0.444951,0.556030,0.317463,0.713479,1.283166,1.0,0.070057,1.549518,0.397582,0.464454,0.354638,0.642212
176,frozenset({fresh fruits}),frozenset({packaged vegetables fruits}),0.556030,0.367935,0.270015,0.485612,1.319832,1.0,0.065432,1.228772,0.545821,0.412899,0.186179,0.609739
175,frozenset({packaged vegetables fruits}),frozenset({fresh fruits}),0.367935,0.556030,0.270015,0.733866,1.319832,1.0,0.065432,1.668223,0.383391,0.412899,0.400560,0.609739
235,frozenset({fresh vegetables}),frozenset({packaged vegetables fruits}),0.444951,0.367935,0.234746,0.527577,1.433888,1.0,0.071033,1.337923,0.545169,0.406037,0.252573,0.582794
234,frozenset({packaged vegetables fruits}),frozenset({fresh vegetables}),0.367935,0.444951,0.234746,0.638010,1.433888,1.0,0.071033,1.533326,0.478741,0.406037,0.347823,0.582794
198,frozenset({yogurt}),frozenset({fresh fruits}),0.261842,0.556030,0.186481,0.712190,1.280848,1.0,0.040889,1.542579,0.297046,0.295349,0.351735,0.523784
199,frozenset({fresh fruits}),frozenset({yogurt}),0.556030,0.261842,0.186481,0.335379,1.280848,1.0,0.040889,1.110646,0.493878,0.295349,0.099623,0.523784
167,frozenset({milk}),frozenset({fresh fruits}),0.241653,0.556030,0.163043,0.674700,1.213423,1.0,0.028677,1.364801,0.231932,0.256906,0.267292,0.483963
172,frozenset({packaged cheese}),frozenset({fresh fruits}),0.231252,0.556030,0.155666,0.673147,1.210630,1.0,0.027083,1.358315,0.226321,0.246457,0.263794,0.476553


## 8. Time-based extension

In [16]:
# Attach order-time information to the sampled orders only.
orders_subset = orders[orders["order_id"].isin(sampled_order_ids)].copy()

def time_bucket(hour):
    if 5 <= hour < 11:
        return "morning"
    elif 11 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 22:
        return "evening"
    else:
        return "night"

orders_subset["time_of_day"] = orders_subset["order_hour_of_day"].apply(time_bucket)
orders_subset.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,time_of_day
29,2037211,3,4,2,18,20.0,evening
83,1224907,10,1,2,14,NaN,afternoon
104,2560699,13,2,0,11,8.0,afternoon
173,603534,17,14,6,17,5.0,evening
196,2831726,17,37,2,12,13.0,afternoon


In [17]:
# Add the time-of-day label as an extra item in each transaction.
order_time_map = orders_subset.set_index("order_id")["time_of_day"].to_dict()
transactions_time = transactions.copy()

for order_id in transactions_time.index:
    transactions_time.loc[order_id].append(order_time_map[order_id])

transactions_time.iloc[:5]

order_id
47     [fresh fruits, hot dogs bacon sausage, package...
322    [fresh herbs, fresh vegetables, nuts seeds dri...
360    [baking ingredients, condiments, fresh dips ta...
393    [breakfast bars pastries, candy chocolate, fre...
441    [asian foods, body lotions soap, cleaning prod...
Name: aisle, dtype: object

In [18]:
# Encode the time-extended transactions.
te_time = TransactionEncoder()
te_array_time = te_time.fit(transactions_time).transform(transactions_time)
df_transactions_time = pd.DataFrame(te_array_time, columns=te_time.columns_)

df_transactions_time.head()

,afternoon,air fresheners candles,asian foods,baby accessories,baby bath body care,baby food formula,bakery desserts,baking ingredients,baking supplies decor,beauty,...,spreads,tea,tofu meat alternatives,tortillas flat bread,trail mix snack mix,trash bags liners,vitamins supplements,water seltzer sparkling water,white wines,yogurt
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,True,False,False,False,False,False,False,True,False,False,...,False,False,True,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,True,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [19]:
# Run Apriori again on the time-extended transactions.
frequent_itemsets_time = apriori(
    df_transactions_time,
    min_support=0.01,
    use_colnames=True
)

rules_time = association_rules(
    frequent_itemsets_time,
    metric="confidence",
    min_threshold=0.3
)

In [20]:
# Filter rules that involve time-of-day categories.
time_keywords = ["morning", "afternoon", "evening", "night"]

rules_time_filtered = rules_time[
    rules_time["antecedents"].apply(lambda x: any(t in x for t in time_keywords)) |
    rules_time["consequents"].apply(lambda x: any(t in x for t in time_keywords))
]

rules_time_filtered.sort_values("lift", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
704,"frozenset({pasta sauce, afternoon})",frozenset({dry pasta}),0.031877,0.070518,0.010626,0.333333,4.726939,1.0,0.008378,1.394223,0.814408,0.115787,0.282755,0.242007
8863,"frozenset({lunch meat, afternoon, fresh vegeta...","frozenset({packaged vegetables fruits, package...",0.030855,0.115370,0.010074,0.326490,2.829928,1.0,0.006514,1.313462,0.667221,0.073991,0.238653,0.206904
8861,"frozenset({packaged vegetables fruits, afterno...","frozenset({packaged cheese, fresh vegetables})",0.026891,0.134967,0.010074,0.374620,2.775648,1.0,0.006445,1.383212,0.657402,0.066370,0.277045,0.224630
8588,"frozenset({fresh fruits, afternoon, other crea...","frozenset({packaged cheese, fresh vegetables})",0.028628,0.134967,0.010421,0.364026,2.697152,1.0,0.006558,1.360170,0.647783,0.068036,0.264798,0.220620
8760,"frozenset({packaged vegetables fruits, afterno...","frozenset({packaged cheese, fresh fruits})",0.026891,0.155666,0.011198,0.416413,2.675038,1.0,0.007012,1.446801,0.643477,0.065347,0.308820,0.244174
10382,"frozenset({yogurt, fresh fruits, afternoon, pa...","frozenset({milk, fresh vegetables})",0.033226,0.123973,0.010994,0.330873,2.668910,1.0,0.006874,1.309209,0.646806,0.075192,0.236180,0.209775
10383,"frozenset({packaged cheese, fresh fruits, afte...","frozenset({yogurt, fresh vegetables})",0.028873,0.142834,0.010994,0.380750,2.665687,1.0,0.006869,1.384201,0.643440,0.068404,0.277562,0.228859
10385,"frozenset({yogurt, afternoon, milk, fresh vege...","frozenset({packaged cheese, fresh fruits})",0.026626,0.155666,0.010994,0.412893,2.652425,1.0,0.006849,1.438126,0.640028,0.064178,0.304651,0.241758
8762,"frozenset({lunch meat, fresh fruits, afternoon})","frozenset({packaged vegetables fruits, package...",0.036740,0.115370,0.011198,0.304783,2.641777,1.0,0.006959,1.272451,0.645171,0.079466,0.214115,0.200921
10313,"frozenset({packaged vegetables fruits, package...","frozenset({yogurt, fresh vegetables})",0.028812,0.142834,0.010646,0.369504,2.586948,1.0,0.006531,1.359510,0.631643,0.066125,0.264441,0.222019
